#### Gradio 기본 UI 컴포넌트 이해하기
* 각 예제는 **셀 단위로 독립 실행**됩니다. 새 예제를 실행하면 이전 서버가 자동으로 종료되어 포트(7860)가 해제됩니다.
* 패턴: `build_xxx()`에서 UI를 **정의**하고, `launch_demo(build_xxx())`로 **실행**합니다.

In [ ]:
import gradio as gr
print(gr.__version__)

##### 0. 공통 헬퍼: 셀별 실행과 포트 해제
* `launch_demo(demo)`: 기존 서버를 먼저 닫고 새로 실행합니다. (`prevent_thread_lock=True`라서 셀이 블로킹되지 않습니다)
* `close_demo()`: 실행 중인 Gradio 서버를 모두 종료해 포트를 해제합니다.
* `is_port_open()`: 포트 점유 여부를 확인합니다.

In [ ]:
import socket
import gradio as gr

GRADIO_HOST = "127.0.0.1"   # 로컬에서만 접속 가능
GRADIO_PORT = 7860          # 모든 예제가 공유하는 포트

def is_port_open(port: int = GRADIO_PORT, host: str = GRADIO_HOST) -> bool:
    """해당 포트에서 서버가 응답하면 True(점유 중), 아니면 False(해제됨)를 반환한다."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def close_demo() -> None:
    """실행 중인 모든 Gradio 서버를 종료하여 포트를 해제한다."""
    gr.close_all()

def launch_demo(demo: gr.Blocks, port: int = GRADIO_PORT) -> None:
    """이전 서버를 종료한 뒤 demo를 실행한다. 셀을 다시 실행해도 포트 충돌이 없다."""
    close_demo()
    demo.launch(server_name=GRADIO_HOST, server_port=port, prevent_thread_lock=True)

print("헬퍼 준비 완료. 포트 점유 여부:", is_port_open())

##### 1. “Hello World” example
* `gr.Interface`: 함수(`fn`) + 입력(`inputs`) + 출력(`outputs`)만 지정하는 가장 간단한 방식입니다.
* 입력/출력은 문자열 단축형(`"text"`)이나 컴포넌트 객체(`gr.Textbox`)로 지정합니다.

In [ ]:
def greet(name: str) -> str:
    """이름을 받아 인사말을 반환한다."""
    return f"Hello, {name}!"

def build_hello() -> gr.Interface:
    """가장 단순한 Interface 예제를 만든다."""
    return gr.Interface(
        fn=greet,
        inputs=gr.Textbox(label="이름"),
        outputs=gr.Textbox(label="인사말"),
    )

launch_demo(build_hello())

##### 2. Input and output components
* 컴포넌트의 `label` 파라미터가 컴포넌트 좌측 상단에 표시됩니다.
* `inputs` 리스트의 순서가 함수 인자의 순서와 일치합니다.
* 여러 개를 출력하려면 함수에서 `,`로 구분해 return하고, `outputs`에 같은 개수의 컴포넌트를 지정합니다.
* 기본 입력 타입: `Image` → numpy 배열, `Audio` → `(샘플레이트, numpy 배열)` 튜플. 입력이 없으면 `None`입니다.

In [ ]:
def process_inputs(text, image, audio, number, option):
    """여러 입력을 받아 요약 문자열과 이미지/오디오 정보를 반환한다. (입력이 비어 있으면 None 처리)"""
    image_info = f"{image.shape}" if image is not None else "없음"
    audio_info = f"{audio[0]}Hz, {len(audio[1])} samples" if audio is not None else "없음"
    summary = f"text={text!r}, number={number}, option={option}"
    return summary, f"이미지: {image_info} / 오디오: {audio_info}"

def build_io() -> gr.Interface:
    """다양한 입력 컴포넌트와 복수 출력을 가진 Interface를 만든다."""
    return gr.Interface(
        fn=process_inputs,
        inputs=[
            gr.Textbox(label="Enter Text"),
            gr.Image(label="Upload Image"),
            gr.Audio(label="Upload Audio"),
            gr.Slider(0, 100, value=50, label="Choose a number"),
            gr.Dropdown(["Streamlit", "Taipy", "Gradio"], value="Gradio", label="Select a UI library"),
        ],
        outputs=[
            gr.Textbox(label="요약"),
            gr.Textbox(label="이미지/오디오 정보"),
        ],
    )

launch_demo(build_io())

##### 3. Row 와 Column
* `gr.Blocks`: 레이아웃을 자유롭게 구성하는 저수준 API입니다. (`Interface`는 `Blocks`의 하위 클래스)
* `gr.Row()`: 컴포넌트를 가로로 배치합니다.
* `gr.Column(scale=..., min_width=...)`: 세로로 배치하며, `scale`로 Row 안에서의 너비 비율을 정합니다.

In [ ]:
import numpy as np

def build_layout() -> gr.Blocks:
    """Row/Column 레이아웃 예제를 만든다. (외부 URL 대신 생성한 그라데이션 이미지 사용)"""
    gradient = np.tile(np.linspace(0, 255, 300, dtype=np.uint8), (150, 1))
    gradient = np.stack([gradient] * 3, axis=-1)   # (높이, 너비) -> (높이, 너비, RGB)

    with gr.Blocks() as demo:
        with gr.Row():                       # 1번째 Row: 3개 컴포넌트를 가로 배치
            gr.Textbox(label="Row-1 Textbox")
            gr.Textbox(label="Row-2 Textbox")
            gr.Dropdown(["a", "b", "c"], label="Row-3 Dropdown")
        with gr.Row():                       # 2번째 Row: 왼쪽 1 : 오른쪽 2 비율
            with gr.Column(scale=1, min_width=300):
                gr.Textbox(label="prompt 1")
                gr.Textbox(label="prompt 2")
                gr.Button("Between")
                gr.Textbox(label="prompt 3")
                gr.Textbox(label="prompt 4")
            with gr.Column(scale=2, min_width=300):
                gr.Image(value=gradient, label="Image")
                gr.Button("Go")
    return demo

launch_demo(build_layout())

##### 4. Tabs 와 Accordions
* `gr.Tab`: 화면을 탭으로 분리합니다.
* `gr.Accordion`: 접고 펼 수 있는 영역입니다. (`open=False`면 접힌 상태로 시작)
* 컴포넌트를 함수와 연결하려면 이벤트 리스너를 사용합니다: `버튼.click(fn, inputs, outputs)`

In [ ]:
def flip_text(x: str) -> str:
    """문자열을 뒤집는다."""
    return x[::-1]

def flip_image(x):
    """이미지를 좌우 반전한다. 이미지가 없으면 None을 반환한다."""
    return None if x is None else np.fliplr(x)

def build_tabs() -> gr.Blocks:
    """Tab + Accordion + 이벤트 리스너 예제를 만든다."""
    with gr.Blocks() as demo:
        gr.Markdown("Flip text or image files using this demo.")
        with gr.Tab("Flip Text"):
            text_input = gr.Textbox(label="입력")
            text_output = gr.Textbox(label="출력")
            text_button = gr.Button("Flip")
        with gr.Tab("Flip Image"):
            with gr.Row():
                image_input = gr.Image(label="입력")
                image_output = gr.Image(label="출력")
            image_button = gr.Button("Flip")

        with gr.Accordion("Open for More!", open=False):
            gr.Markdown("Look at me...")
            gr.Slider(0, 1, value=0.1, step=0.1, interactive=True, label="Slide me")

        # 이벤트 연결: 버튼 클릭 시 fn(inputs) -> outputs
        text_button.click(flip_text, inputs=text_input, outputs=text_output)
        image_button.click(flip_image, inputs=image_input, outputs=image_output)
    return demo

launch_demo(build_tabs())

##### 5. 이벤트 리스너 (click / change / submit)
* `click`: 버튼 클릭 시
* `change`: 값이 바뀔 때마다 (실시간 반응)
* `submit`: Textbox에서 Enter를 눌렀을 때
* 하나의 함수를 여러 이벤트에 연결할 수 있고, `inputs`에 여러 컴포넌트를 리스트로 전달합니다.

In [ ]:
def make_greeting(name: str, shout: bool, tone: str) -> str:
    """이름, 대문자 여부, 말투를 반영한 인사말을 만든다."""
    if not name:
        return "이름을 입력하세요."
    text = f"{name}님, 안녕하세요." if tone == "존댓말" else f"{name}, 안녕."
    return text.upper() if shout else text

def build_events() -> gr.Blocks:
    """여러 입력 컴포넌트와 이벤트 리스너를 연결한 예제를 만든다."""
    with gr.Blocks() as demo:
        gr.Markdown("### 이벤트 리스너: click / change / submit")
        with gr.Row():
            with gr.Column():
                name = gr.Textbox(label="이름 (Enter로 submit)")
                shout = gr.Checkbox(label="대문자로 변환")
                tone = gr.Radio(["존댓말", "반말"], value="존댓말", label="말투")
                button = gr.Button("인사하기", variant="primary")
            with gr.Column():
                greeting = gr.Textbox(label="결과")
                counter = gr.Textbox(label="글자 수 (change 이벤트로 실시간 갱신)")

        button.click(make_greeting, inputs=[name, shout, tone], outputs=greeting)
        name.submit(make_greeting, inputs=[name, shout, tone], outputs=greeting)
        name.change(lambda t: f"{len(t)}자", inputs=name, outputs=counter)
    return demo

launch_demo(build_events())

##### 6. 서버 종료와 포트 해제 확인
* 실습이 끝나면 아래 셀을 실행해 서버를 종료하고 포트가 해제되었는지 확인합니다.

In [ ]:
close_demo()
print("포트 점유 여부:", is_port_open())   # False면 해제 완료